# 20b. 최종 모델 **테스트** (저장된 모델 사용)

`20a_final_train.ipynb`에서 저장한 `.pkl`을 불러와 labeled에 점수 → 평가
- 모델은 labeled를 학습에 쓴 적 없음 → fold 없이 labeled 전체가 테스트셋
- 파생변수 기준값도 **학습 때 저장한 값**을 그대로 사용 (테스트 데이터로 다시 계산하지 않음)
- 라벨은 채점에만 사용

In [ ]:
import pandas as pd
from final_model import test, compare_with_cache
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)

R = test()

## 0. 재현 확인 (18번 수치와 같은지)

In [ ]:
display(R['check'])

## 1. 평가 ① 공식 2차 · 행 단위 (팀원 평가 틀)
- **AP**: 주 지표 / **무작위 평균 대비**·**유의**: 무작위 순위 1,000번의 97.5%를 넘으면 O
- **상위 k%**: 점수 상위 k% 행만 검사할 때 잡히는 불량 행
- **F1@5%·10%**: 상위 k%를 불량 판정 / **F1(최적,참고)**: 정답 보고 고른 임계값 → 참고용
- **상한 F1**: LH/RH 같은 입력 행은 같은 판정 → 어떤 모델도 못 넘는 값

In [ ]:
display(R['frame1'].round(3))

## 2. 평가 ② labeled 전체 · 샷 단위 · 초기허용불량 제외
- **AP / AP_lo~AP_hi**: AP와 부트스트랩 95% 구간, **AP/불량률** = 무작위 대비 배수
- **Run별 / 전체 상위 k%**, **유형별 AP** (가스·미성형)

In [ ]:
display(R['frame2'].round(3))

## 3. 라벨 없이 정한 임계값의 경보 성능 (② 기준)
**경보 비율 ≈ 1 − q** 이면 임계값 보정이 잘 된 것

In [ ]:
display(R['threshold'].round(3))

## 4. 경보 샷 확인 (q = 0.999)

In [ ]:
for p, f in R['scores'].items():
    a = f[(f['pct'] >= 0.999) & (f['split'] != '제외') & (f['Reason'] != '초기허용불량')]
    print(p.upper(), f'경보 {len(a)}샷')
    display(a[['TimeStamp', 'run_id', 'PassOrFail', 'Reason', 'score', 'pct']].round(4))

## 5. (선택) 기존 파생변수 캐시와 비교

In [ ]:
for p in ['cn7', 'rg3']:
    print(p); display(compare_with_cache(R, p))